In [1]:

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.model_config import get_config
from src.annual_loss_simulation import simulate_annual_losses
from src.incident_simulation import simulate_incidents

cfg = get_config()

print("Project root:", PROJECT_ROOT)
print("Configuration loaded successfully.")


Project root: /Users/shreyagokhale/Desktop/Docs/Documents/Research projects_cybersecurity/novaflow-fair-crq
Configuration loaded successfully.


In [2]:
#Generate validation samples


annual_validation = simulate_annual_losses(
    n_years=100_000,
    seed=20261103,
    config=cfg,
)

incident_validation = simulate_incidents(
    n_incidents=100_000,
    seed=20261104,
    config=cfg,
    include_details=True,
)

print("Annual observations:", len(annual_validation))
print("Incident observations:", len(incident_validation))

print("\nAnnual columns:")
print(annual_validation.columns.tolist())

print("\nIncident columns:")
print(incident_validation.columns.tolist())


Annual observations: 100000
Incident observations: 100000

Annual columns:
['year_id', 'campaign_count', 'successful_incident_count', 'primary_loss_usd', 'secondary_response_planning_loss_usd', 'secondary_fines_judgement_loss_usd', 'secondary_reputation_loss_usd', 'total_annual_loss_usd', 'maximum_single_event_loss_usd']

Incident columns:
['incident_id', 'technical_severity', 'affected_customers', 'primary_loss_usd', 'secondary_response_planning_loss_usd', 'secondary_fines_judgement_loss_usd', 'secondary_reputation_loss_usd', 'total_incident_loss_usd', 'indicator_public_visibility', 'indicator_notification', 'indicator_regulatory_engagement', 'indicator_external_pr_agency', 'indicator_judgment_or_settlement', 'indicator_regulatory_fine', 'primary_technical_response_hours', 'primary_management_response_hours', 'primary_executive_response_hours', 'primary_technical_response_cost_usd', 'primary_management_response_cost_usd', 'primary_executive_response_cost_usd', 'primary_external_dfir_c

In [3]:
#Validate incident frequency


campaign_rate = 3.0
alpha = 0.36
beta = 4.48

expected_success_probability = alpha / (alpha + beta)
expected_lef = campaign_rate * expected_success_probability
expected_any_incident = 1 - np.exp(-expected_lef)

print("THEORETICAL FREQUENCY TARGETS")
print("-" * 45)

print(
    f"Expected campaign rate: "
    f"{campaign_rate:.6f}"
)

print(
    f"Expected success probability: "
    f"{expected_success_probability:.6%}"
)

print(
    f"Expected annual LEF: "
    f"{expected_lef:.6f}"
)

print(
    f"Expected probability of >=1 incident: "
    f"{expected_any_incident:.6%}"
)

print("\nAVAILABLE ANNUAL DATA COLUMNS")
print(annual_validation.columns.tolist())


THEORETICAL FREQUENCY TARGETS
---------------------------------------------
Expected campaign rate: 3.000000
Expected success probability: 7.438017%
Expected annual LEF: 0.223140
Expected probability of >=1 incident: 19.999756%

AVAILABLE ANNUAL DATA COLUMNS
['year_id', 'campaign_count', 'successful_incident_count', 'primary_loss_usd', 'secondary_response_planning_loss_usd', 'secondary_fines_judgement_loss_usd', 'secondary_reputation_loss_usd', 'total_annual_loss_usd', 'maximum_single_event_loss_usd']


In [5]:
# Validate annual incident frequency

campaign_col = "campaign_count"
incident_col = "successful_incident_count"

observed_campaign_rate = annual_validation[campaign_col].mean()
observed_lef = annual_validation[incident_col].mean()

observed_any_incident = (
    annual_validation[incident_col] > 0
).mean()

frequency_comparison = pd.DataFrame([
    {
        "Quantity": "Campaigns per year",
        "Theoretical": campaign_rate,
        "Simulated": observed_campaign_rate,
    },
    {
        "Quantity": "Successful incidents per year",
        "Theoretical": expected_lef,
        "Simulated": observed_lef,
    },
    {
        "Quantity": "Probability of >=1 incident",
        "Theoretical": expected_any_incident,
        "Simulated": observed_any_incident,
    },
])

frequency_comparison["Absolute difference"] = (
    frequency_comparison["Simulated"]
    - frequency_comparison["Theoretical"]
).abs()

display(
    frequency_comparison.style.format({
        "Theoretical": "{:.6f}",
        "Simulated": "{:.6f}",
        "Absolute difference": "{:.6f}",
    })
)


,Quantity,Theoretical,Simulated,Absolute difference
0,Campaigns per year,3.000000,2.990150,0.009850
1,Successful incidents per year,0.223140,0.225400,0.002260
2,Probability of >=1 incident,0.199998,0.201130,0.001132


In [6]:

#Validate calibrated event probabilities

probability_targets = {
    "public_visibility": 0.30,
    "notification": 0.75,
    "regulatory_engagement": 0.10,
    "external_pr_agency": 0.30,
    "regulatory_fine": 0.05,
    "judgment_or_settlement": 0.05,
}

rows = []

for name, target in probability_targets.items():

    column = f"indicator_{name}"

    if column not in incident_validation.columns:
        raise KeyError(
            f"Missing required indicator: {column}"
        )

    observed = incident_validation[column].mean()

    n = len(incident_validation)

    # Standard error under the target Bernoulli probability
    se = np.sqrt(target * (1 - target) / n)

    z_score = (
        (observed - target) / se
        if se > 0
        else np.nan
    )

    rows.append({
        "Indicator": name,
        "Target": target,
        "Observed": observed,
        "Difference": observed - target,
        "Z-score": z_score,
        "Within ±3 SE": abs(z_score) <= 3,
    })

probability_validation = pd.DataFrame(rows)

display(
    probability_validation.style.format({
        "Target": "{:.3%}",
        "Observed": "{:.3%}",
        "Difference": "{:+.3%}",
        "Z-score": "{:+.2f}",
    })
)


,Indicator,Target,Observed,Difference,Z-score,Within ±3 SE
0,public_visibility,30.000%,30.298%,+0.298%,+2.06,True
1,notification,75.000%,75.215%,+0.215%,+1.57,True
2,regulatory_engagement,10.000%,9.873%,-0.127%,-1.34,True
3,external_pr_agency,30.000%,29.938%,-0.062%,-0.43,True
4,regulatory_fine,5.000%,4.983%,-0.017%,-0.25,True
5,judgment_or_settlement,5.000%,5.055%,+0.055%,+0.80,True


In [7]:
#Validate the regulatory fine gate


regulator = incident_validation[
    "indicator_regulatory_engagement"
].astype(bool)

fine = incident_validation[
    "indicator_regulatory_fine"
].astype(bool)

violations = fine & ~regulator

print("Regulatory engagement rate:", f"{regulator.mean():.3%}")
print("Unconditional fine rate:", f"{fine.mean():.3%}")

print(
    "Fine probability given regulatory engagement:",
    f"{fine[regulator].mean():.3%}"
)

print(
    "Fine without regulatory engagement:",
    int(violations.sum())
)

assert violations.sum() == 0, (
    "Regulatory fine gate violated."
)

print("\nPASS: All fines occur within regulatory engagement.")


Regulatory engagement rate: 9.873%
Unconditional fine rate: 4.983%
Fine probability given regulatory engagement: 50.471%
Fine without regulatory engagement: 0

PASS: All fines occur within regulatory engagement.


In [8]:

#Inspect customer and loss-related columns
keywords = [
    "customer",
    "churn",
    "deal",
    "opportun",
    "severity",
    "loss",
]

matching_columns = [
    col
    for col in incident_validation.columns
    if any(
        keyword in col.lower()
        for keyword in keywords
    )
]

print("Customer, severity, and loss-related columns:\n")

for col in matching_columns:
    print(col)

print("\nAffected-customer distribution summary:\n")

display(
    incident_validation[
        "affected_customers"
    ].describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
)


Customer, severity, and loss-related columns:

technical_severity
affected_customers
primary_loss_usd
secondary_response_planning_loss_usd
secondary_fines_judgement_loss_usd
secondary_reputation_loss_usd
total_incident_loss_usd
response_legal_customer_notification_active
response_legal_customer_notification_outside_hours
response_legal_customer_notification_internal_hours
fines_regulatory_fine_loss_usd
fines_judgment_or_settlement_loss_usd
reputation_affected_customers
reputation_unaffected_customers
reputation_affected_churn_probability
reputation_unaffected_churn_probability
reputation_lost_deal_probability
reputation_affected_churn_count
reputation_unaffected_churn_count
reputation_total_churn_count
reputation_otherwise_won_opportunities
reputation_lost_deals_count
reputation_affected_churn_loss_usd
reputation_unaffected_churn_loss_usd
reputation_lost_deals_loss_usd

Affected-customer distribution summary:



count    100000.000000
mean       1553.112280
std         510.848106
min         312.000000
5%          732.000000
25%        1167.000000
50%        1542.000000
75%        1927.000000
95%        2413.000000
max        2962.000000
Name: affected_customers, dtype: float64

In [9]:
#Validate customer-weighted churn and lost-deal rates

df = incident_validation

affected_exposure = df["reputation_affected_customers"].sum()
unaffected_exposure = df["reputation_unaffected_customers"].sum()
opportunity_exposure = df["reputation_otherwise_won_opportunities"].sum()

assert min(
    affected_exposure,
    unaffected_exposure,
    opportunity_exposure
) > 0, "At least one exposure denominator is zero."

targets = {
    "Affected-customer churn": 0.002,
    "Unaffected-customer churn": 0.0002,
    "Lost-deal probability": 0.01,
}

observed_counts = {
    "Affected-customer churn":
        df["reputation_affected_churn_count"].sum(),
    "Unaffected-customer churn":
        df["reputation_unaffected_churn_count"].sum(),
    "Lost-deal probability":
        df["reputation_lost_deals_count"].sum(),
}

exposures = {
    "Affected-customer churn": affected_exposure,
    "Unaffected-customer churn": unaffected_exposure,
    "Lost-deal probability": opportunity_exposure,
}

rows = []

for name, target in targets.items():

    count = observed_counts[name]
    exposure = exposures[name]
    observed = count / exposure

    # Approximate binomial SE for a diagnostic comparison.
    # Shared incident-level probabilities can create
    # overdispersion, so this is not a formal significance test.
    se = np.sqrt(target * (1 - target) / exposure)

    rows.append({
        "Metric": name,
        "Target": target,
        "Observed": observed,
        "Absolute difference": abs(observed - target),
        "Approx. SE": se,
        "Within 3 approx. SE": abs(observed - target) <= 3 * se,
    })

churn_validation = pd.DataFrame(rows)

display(
    churn_validation.style.format({
        "Target": "{:.4%}",
        "Observed": "{:.4%}",
        "Absolute difference": "{:.4%}",
        "Approx. SE": "{:.6%}",
    })
)


,Metric,Target,Observed,Absolute difference,Approx. SE,Within 3 approx. SE
0,Affected-customer churn,0.2000%,0.2009%,0.0009%,0.000358%,True
1,Unaffected-customer churn,0.0200%,0.0201%,0.0001%,0.000118%,True
2,Lost-deal probability,1.0000%,1.0052%,0.0052%,0.005245%,True


In [10]:

# Verify customer counts and reputation loss

df = incident_validation

customer_total = 3000
margin = 210_000

checks = {
    "Affected + unaffected = 3,000":
        (
            df["reputation_affected_customers"]
            + df["reputation_unaffected_customers"]
            == customer_total
        ).all(),

    "Affected churn <= affected customers":
        (
            df["reputation_affected_churn_count"]
            <= df["reputation_affected_customers"]
        ).all(),

    "Unaffected churn <= unaffected customers":
        (
            df["reputation_unaffected_churn_count"]
            <= df["reputation_unaffected_customers"]
        ).all(),

    "Lost deals <= otherwise-won opportunities":
        (
            df["reputation_lost_deals_count"]
            <= df["reputation_otherwise_won_opportunities"]
        ).all(),

    "Affected churn loss correct":
        np.allclose(
            df["reputation_affected_churn_loss_usd"],
            df["reputation_affected_churn_count"] * margin,
        ),

    "Unaffected churn loss correct":
        np.allclose(
            df["reputation_unaffected_churn_loss_usd"],
            df["reputation_unaffected_churn_count"] * margin,
        ),

    "Lost-deal loss correct":
        np.allclose(
            df["reputation_lost_deals_loss_usd"],
            df["reputation_lost_deals_count"] * margin,
        ),

    "Total reputation loss correct":
        np.allclose(
            df["secondary_reputation_loss_usd"],
            df["reputation_affected_churn_loss_usd"]
            + df["reputation_unaffected_churn_loss_usd"]
            + df["reputation_lost_deals_loss_usd"],
        ),
}

checks_df = pd.DataFrame([
    {"Check": name, "Passed": bool(result)}
    for name, result in checks.items()
])

display(checks_df)

assert checks_df["Passed"].all(), (
    "At least one reputation accounting check failed."
)

print("All reputation accounting checks passed.")


,Check,Passed
0,"Affected + unaffected = 3,000",True
1,Affected churn <= affected customers,True
2,Unaffected churn <= unaffected customers,True
3,Lost deals <= otherwise-won opportunities,True
4,Affected churn loss correct,True
5,Unaffected churn loss correct,True
6,Lost-deal loss correct,True
7,Total reputation loss correct,True


All reputation accounting checks passed.


In [11]:

#Verify total loss equals sum of four components

components = [
    "primary_loss_usd",
    "secondary_response_planning_loss_usd",
    "secondary_fines_judgement_loss_usd",
    "secondary_reputation_loss_usd",
]

reconstructed_total = incident_validation[
    components
].sum(axis=1)

reported_total = incident_validation[
    "total_incident_loss_usd"
]

difference = reported_total - reconstructed_total

print(
    "Maximum absolute accounting difference:",
    f"${np.abs(difference).max():,.8f}"
)

assert np.allclose(
    reported_total,
    reconstructed_total,
    rtol=1e-10,
    atol=1e-6,
)

assert (
    incident_validation[
        components + ["total_incident_loss_usd"]
    ] >= 0
).all().all()

print("PASS: Incident losses reconcile correctly.")

component_means = (
    incident_validation[components]
    .mean()
    .rename("Mean loss per incident")
    .reset_index()
    .rename(columns={"index": "Loss component"})
)

component_means["Share of expected incident loss"] = (
    component_means["Mean loss per incident"]
    / reported_total.mean()
)

display(
    component_means.style.format({
        "Mean loss per incident": "${:,.0f}",
        "Share of expected incident loss": "{:.2%}",
    })
)


Maximum absolute accounting difference: $0.00000000
PASS: Incident losses reconcile correctly.


,Loss component,Mean loss per incident,Share of expected incident loss
0,primary_loss_usd,"$147,381",11.92%
1,secondary_response_planning_loss_usd,"$187,884",15.19%
2,secondary_fines_judgement_loss_usd,"$109,398",8.85%
3,secondary_reputation_loss_usd,"$792,082",64.05%
